# Early Warning Experiments

Ce notebook reprend le script `early_warning_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Entraine directement des modeles d'alerte precoce causale avant entree physique.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Train models directly for causal early warning before physical entry.
- Run par defaut : `runs/exp_093_early_warning_direct`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "early_warning_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import time
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from ml_pipeline import ROOT, RUNS_DIR, draw_overlay, load_dataset, read_frame, safe_auc, write_json, zone_polygon
from sequence_experiments import make_model, make_run_dir, set_seed


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `load_sequence_data`

Cette cellule definit `load_sequence_data`. Elle prepare une partie du script.

In [ ]:
def load_sequence_data(sequence_run, base_run):
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    meta = pd.read_csv(sequence_run / "features" / "sequence_index.csv")
    entry = pd.read_csv(base_run / "features" / "entry_times.csv")
    entry_small = entry[["video_id", "event_type", "target_source", "body_part"]].copy()
    meta = meta.merge(entry_small, on="video_id", how="left")
    return X_raw, meta


## Fonction `normalize_from_train`

Cette cellule definit `normalize_from_train`. Elle prepare une partie du script.

In [ ]:
def normalize_from_train(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


## Fonction `build_early_target`

Cette cellule definit `build_early_target`. Elle prepare une partie du script.

In [ ]:
def build_early_target(meta, min_early_s, max_early_s, mode):
    tte = pd.to_numeric(meta["time_to_target_s"], errors="coerce")
    is_entry = meta["is_danger_clip"].astype(int).eq(1)
    y = np.zeros(len(meta), dtype=np.float32)
    train_mask = np.ones(len(meta), dtype=bool)

    if mode == "band":
        positive = is_entry & tte.ge(min_early_s) & tte.le(max_early_s)
        too_late = is_entry & tte.ge(0.0) & tte.lt(min_early_s)
        y[positive.to_numpy()] = 1.0
        train_mask[too_late.to_numpy()] = False
    elif mode == "cumulative":
        positive = is_entry & tte.ge(min_early_s) & tte.le(max_early_s)
        too_late = is_entry & tte.ge(0.0) & tte.lt(min_early_s)
        y[positive.to_numpy()] = 1.0
        train_mask[too_late.to_numpy()] = False
    elif mode == "late_penalty":
        positive = is_entry & tte.ge(min_early_s) & tte.le(max_early_s)
        y[positive.to_numpy()] = 1.0
    else:
        raise ValueError(f"Unknown target mode: {mode}")

    return y.reshape(-1, 1), train_mask


## Classe `EarlyDataset`

Cette cellule definit `EarlyDataset`. Elle prepare une partie du script.

In [ ]:
class EarlyDataset(Dataset):
    def __init__(self, X, y, sample_weight, indices, augment=False, seed=42):
        self.X = X
        self.y = y
        self.sample_weight = sample_weight
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        x = self.X[source_idx].copy()
        if self.augment:
            if self.rng.random() < 0.85:
                x += self.rng.normal(0.0, 0.035, size=x.shape).astype(np.float32)
            if self.rng.random() < 0.35:
                n_features = max(1, int(x.shape[1] * self.rng.uniform(0.03, 0.10)))
                cols = self.rng.choice(x.shape[1], size=n_features, replace=False)
                x[:, cols] = 0.0
            if self.rng.random() < 0.35:
                width = int(self.rng.integers(2, max(3, x.shape[0] // 4)))
                start = int(self.rng.integers(0, max(1, x.shape[0] - width + 1)))
                x[start : start + width] = 0.0
        return (
            torch.from_numpy(x.astype(np.float32)),
            torch.from_numpy(self.y[source_idx].astype(np.float32)),
            torch.tensor(self.sample_weight[source_idx], dtype=torch.float32),
        )


## Fonction `weighted_loss`

Cette cellule definit `weighted_loss`. Elle prepare une partie du script.

In [ ]:
def weighted_loss(logits, targets, sample_weight, pos_weight, loss_name, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none").view(-1)
    if loss_name == "focal":
        pt = torch.exp(-bce).clamp(min=1e-6, max=1.0)
        bce = ((1.0 - pt) ** gamma) * bce
    return (bce * sample_weight).sum() / sample_weight.sum().clamp(min=1.0)


## Fonction `predict`

Cette cellule definit `predict`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict(model, X, batch_size, device):
    model.eval()
    rows = []
    for start in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[start : start + batch_size]).to(device)
        rows.append(torch.sigmoid(model(xb)).detach().cpu().numpy().reshape(-1))
    return np.concatenate(rows)


## Fonction `alarm_episodes_hysteresis`

Cette cellule definit `alarm_episodes_hysteresis`. Elle prepare une partie du script.

In [ ]:
def alarm_episodes_hysteresis(times, scores, on_threshold, off_threshold, gap_s=1.0, persistence_windows=2):
    times = [float(t) for t in times]
    scores = [float(s) for s in scores]
    alarms = []
    active = False
    pending = []
    last_active_t = None
    for t, score in zip(times, scores):
        if active:
            if score < off_threshold:
                active = False
                last_active_t = t
            continue
        if score >= on_threshold:
            pending.append(t)
            if len(pending) >= persistence_windows:
                alarms.append(pending[0])
                active = True
                pending = []
        else:
            pending = []
        if last_active_t is not None and t - last_active_t > gap_s:
            last_active_t = None
    if not alarms:
        return []
    episodes = [alarms[0]]
    for t in alarms[1:]:
        if t - episodes[-1] > gap_s:
            episodes.append(t)
    return episodes


## Fonction `evaluate_causal`

Cette cellule definit `evaluate_causal`. Elle prepare une partie du script.

In [ ]:
def evaluate_causal(pred, score_col, threshold, split_name, seed, persistence_windows, early_margin_s, hysteresis=False):
    split_df = pred[(pred["split"] == split_name) & (pred["repeat_seed"] == seed)].copy()
    if split_df.empty:
        return None
    pre = early = miss = fp = danger = 0
    neg_minutes = 0.0
    early_times = []
    for _, group in split_df.groupby("video_id", sort=False):
        group = group.sort_values("time_s")
        if hysteresis:
            alarms = alarm_episodes_hysteresis(
                group["time_s"],
                group[score_col],
                threshold,
                max(0.01, threshold * 0.60),
                persistence_windows=persistence_windows,
            )
        else:
            from ml_pipeline import alarm_episodes

            alarms = alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=persistence_windows)
        is_danger = int(group["is_danger_clip"].max()) == 1
        target_values = pd.to_numeric(group["target_time_s"], errors="coerce").dropna()
        target = float(target_values.iloc[0]) if len(target_values) else math.nan
        if is_danger and not math.isnan(target):
            danger += 1
            pre_alarms = [float(t) for t in alarms if float(t) < target]
            if pre_alarms:
                first = min(pre_alarms)
                lead = target - first
                early_times.append(lead)
                pre += 1
                if lead >= early_margin_s:
                    early += 1
            else:
                miss += 1
        else:
            fp += len(alarms)
            if len(group):
                neg_minutes += max(0.0, float(group["time_s"].max() - group["time_s"].min())) / 60.0
    precision = pre / (pre + fp) if (pre + fp) else np.nan
    pre_recall = pre / danger if danger else np.nan
    early_recall = early / danger if danger else np.nan
    f1 = 2 * precision * pre_recall / (precision + pre_recall) if precision == precision and (precision + pre_recall) > 0 else np.nan
    return {
        "repeat_seed": int(seed),
        "split": split_name,
        "threshold": float(threshold),
        "hysteresis": bool(hysteresis),
        "danger_videos": int(danger),
        "pre_entry_detected": int(pre),
        f"early_{early_margin_s:.1f}s_detected": int(early),
        "missed_entries": int(miss),
        "false_alarm_episodes": int(fp),
        "negative_minutes": float(neg_minutes),
        "pre_entry_recall": float(pre_recall),
        f"early_{early_margin_s:.1f}s_recall": float(early_recall),
        "event_precision": float(precision),
        "event_f1": float(f1),
        "false_alarms_per_min": float(fp / neg_minutes) if neg_minutes > 0 else 0.0,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
        "mean_early_warning_s": float(np.mean(early_times)) if early_times else np.nan,
    }


## Fonction `causal_selection_score`

Cette cellule definit `causal_selection_score`. Elle prepare une partie du script.

In [ ]:
def causal_selection_score(row, early_margin_s):
    return (
        2.0 * row[f"early_{early_margin_s:.1f}s_recall"]
        + 0.8 * row["pre_entry_recall"]
        + 0.7 * row["event_precision"]
        - 0.06 * min(row["false_alarms_per_min"], 20.0)
    )


## Fonction `summarize_causal`

Cette cellule definit `summarize_causal`. Elle prepare une partie du script.

In [ ]:
def summarize_causal(rows, early_margin_s):
    df = pd.DataFrame(rows)
    out = []
    group_cols = ["model", "split", "threshold", "hysteresis"]
    for keys, group in df.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["danger_videos_total"] = int(group["danger_videos"].sum())
        row["pre_entry_detected_total"] = int(group["pre_entry_detected"].sum())
        row[f"early_{early_margin_s:.1f}s_detected_total"] = int(group[f"early_{early_margin_s:.1f}s_detected"].sum())
        row["false_alarm_episodes_total"] = int(group["false_alarm_episodes"].sum())
        for col in ["pre_entry_recall", f"early_{early_margin_s:.1f}s_recall", "event_precision", "event_f1", "false_alarms_per_min", "median_early_warning_s"]:
            row[f"{col}_mean"] = float(pd.to_numeric(group[col], errors="coerce").mean())
            row[f"{col}_std"] = float(pd.to_numeric(group[col], errors="coerce").std(ddof=0))
        out.append(row)
    return pd.DataFrame(out)


## Fonction `train_model`

Cette cellule definit `train_model`. Elle prepare une partie du script.

In [ ]:
def train_model(spec, X, y, train_mask, sample_weight, meta, args, run_dir, device):
    name, kind, loss_name, augment = spec
    set_seed(args.seed)
    usable = train_mask.copy()
    train_idx = np.flatnonzero((meta["split"].to_numpy() == "train") & usable)
    val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")
    train_ds = EarlyDataset(X, y, sample_weight, train_idx, augment=augment, seed=args.seed)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    model = make_model(kind, X.shape[1], X.shape[2], 1).to(device)
    positives = float(y[train_idx].sum())
    negatives = float(len(train_idx) - positives)
    pos_weight = torch.tensor([np.clip(negatives / max(1.0, positives), 1.0, args.max_pos_weight)], dtype=torch.float32, device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    best = {"score": -1e9, "state": None, "epoch": 0}
    history = []
    start = time.perf_counter()
    patience_left = args.patience
    thresholds = [round(x, 2) for x in np.arange(0.05, 1.0, 0.05)]
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb, wb in train_loader:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = weighted_loss(model(xb), yb, wb, pos_weight, loss_name, args.focal_gamma)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_scores = predict(model, X[val_idx], args.batch_size, device)
        val_pred = meta.iloc[val_idx].copy()
        val_pred["repeat_seed"] = args.seed
        val_pred["risk"] = val_scores
        eval_rows = []
        for threshold in thresholds:
            for hysteresis in [False, True]:
                row = evaluate_causal(val_pred, "risk", threshold, "val", args.seed, args.persistence_windows, args.early_margin_s, hysteresis=hysteresis)
                if row:
                    eval_rows.append(row)
        val_eval = pd.DataFrame(eval_rows)
        val_eval["score"] = val_eval.apply(lambda r: causal_selection_score(r, args.early_margin_s), axis=1)
        val_score = float(val_eval["score"].max()) if len(val_eval) else -1e9
        scheduler.step(val_score)
        history.append({"model": name, "epoch": epoch, "train_loss": float(np.mean(losses)), "val_causal_score": val_score, "lr": float(optimizer.param_groups[0]["lr"])})
        if val_score > best["score"] + 1e-5:
            best = {"score": val_score, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "epoch": epoch}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time = time.perf_counter() - start
    torch.save(
        {
            "state_dict": model.state_dict(),
            "model_name": name,
            "kind": kind,
            "loss": loss_name,
            "seq_len": int(X.shape[1]),
            "input_dim": int(X.shape[2]),
            "target": {
                "min_early_s": args.min_early_s,
                "max_early_s": args.max_early_s,
                "early_margin_s": args.early_margin_s,
                "mode": args.target_mode,
            },
            "best_epoch": best["epoch"],
        },
        run_dir / "models" / f"{name}.pt",
    )
    probs = predict(model, X, args.batch_size, device)
    pred = meta.copy()
    pred["repeat_seed"] = args.seed
    pred["risk"] = probs
    pred["early_target"] = y.reshape(-1)
    pred["train_usable"] = train_mask.astype(int)
    pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)
    return model, pred, history, train_time


## Fonction `make_failure_screenshots`

Cette cellule definit `make_failure_screenshots`. Elle prepare une partie du script.

In [ ]:
def make_failure_screenshots(run_dir, pred, selected, limit=16):
    videos, _, _, zones = load_dataset()
    video_by_id = {row["video_id"]: row for row in videos}
    polygon = zone_polygon(zones)
    out_dir = run_dir / "error_review" / "causal_failures"
    out_dir.mkdir(parents=True, exist_ok=True)
    score_col = "risk"
    threshold = float(selected["threshold"])
    hysteresis = bool(selected["hysteresis"])
    saved = 0
    test = pred[pred["split"] == "test"].copy()
    for video_id, group in test.groupby("video_id", sort=False):
        if saved >= limit:
            break
        group = group.sort_values("time_s")
        is_danger = int(group["is_danger_clip"].max()) == 1
        target_values = pd.to_numeric(group["target_time_s"], errors="coerce").dropna()
        target = float(target_values.iloc[0]) if len(target_values) else math.nan
        if hysteresis:
            alarms = alarm_episodes_hysteresis(group["time_s"], group[score_col], threshold, max(0.01, threshold * 0.60), persistence_windows=2)
        else:
            from ml_pipeline import alarm_episodes

            alarms = alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=2)
        label = None
        frame_time = None
        if is_danger and not math.isnan(target):
            pre = [float(t) for t in alarms if float(t) < target]
            if not pre:
                label = "CAUSAL MISS"
                frame_time = target
            elif target - min(pre) < 0.5:
                label = "LATE <0.5s"
                frame_time = min(pre)
        elif alarms:
            label = "FALSE ALARM"
            frame_time = min(alarms)
        if label is None or video_id not in video_by_id:
            continue
        video = video_by_id[video_id]
        frame_idx = int(round(frame_time * float(video["fps"])))
        frame = read_frame(ROOT / video["path"], frame_idx)
        if frame is None:
            continue
        nearest = group.iloc[(group["time_s"] - frame_time).abs().argsort().iloc[0]]
        text = [
            label,
            f"risk={float(nearest['risk']):.3f} thr={threshold:.2f}",
            f"target={target:.2f}s alarm={frame_time:.2f}s",
            video["path"],
        ]
        out = draw_overlay(frame, polygon, text)
        cv2.imwrite(str(out_dir / f"{video_id}_{label.lower().replace(' ', '_').replace('<', 'lt')}.jpg"), out)
        saved += 1
    return saved


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    sequence_run = resolve(args.sequence_run)
    base_run = resolve(args.base_run)
    run_dir = make_run_dir(args.run_name)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    X_raw, meta = load_sequence_data(sequence_run, base_run)
    X, mean, std = normalize_from_train(X_raw, meta)
    y, train_mask = build_early_target(meta, args.min_early_s, args.max_early_s, args.target_mode)

    sample_weight = np.ones(len(meta), dtype=np.float32)
    hard_negative = meta["event_type"].fillna("").eq("near_miss") | ((meta["is_danger_clip"].astype(int).eq(1)) & (pd.to_numeric(meta["time_to_target_s"], errors="coerce") > args.max_early_s))
    sample_weight[hard_negative.to_numpy()] *= args.hard_negative_weight
    sample_weight[~train_mask] = 0.0

    specs = []
    for item in args.specs:
        name, kind, loss_name, aug = item.split(":")
        specs.append((name, kind, loss_name, aug == "aug"))

    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(sequence_run),
            "base_run": str(base_run),
            "target_mode": args.target_mode,
            "min_early_s": args.min_early_s,
            "max_early_s": args.max_early_s,
            "early_margin_s": args.early_margin_s,
            "hard_negative_weight": args.hard_negative_weight,
            "specs": args.specs,
            "device": str(device),
        },
    )
    write_json(
        run_dir / "metrics" / "early_target_audit.json",
        {
            "rows": int(len(meta)),
            "train_usable_rows": int(train_mask.sum()),
            "positive_rows": int(y.sum()),
            "hard_negative_rows": int(hard_negative.sum()),
            "split_positive_rows": {split: int(y[meta["split"].eq(split).to_numpy()].sum()) for split in ["train", "val", "test"]},
            "ignored_too_late_rows": int((~train_mask).sum()),
        },
    )
    np.savez_compressed(run_dir / "features" / "early_sequence_normalizer.npz", mean=mean, std=std)

    all_history = []
    all_causal = []
    all_predictions = []
    thresholds = [round(x, 2) for x in np.arange(0.05, 1.0, 0.05)]
    for spec in specs:
        model_name = spec[0]
        print(f"training {model_name} on {device}")
        model, pred, history, train_time = train_model(spec, X, y, train_mask, sample_weight, meta, args, run_dir, device)
        for row in history:
            row["train_time_s"] = train_time
        all_history.extend(history)
        pred["model"] = model_name
        all_predictions.append(pred)
        for threshold in thresholds:
            for hysteresis in [False, True]:
                for split in ["train", "val", "test"]:
                    row = evaluate_causal(pred, "risk", threshold, split, args.seed, args.persistence_windows, args.early_margin_s, hysteresis=hysteresis)
                    if row:
                        row["model"] = model_name
                        row["selection_score"] = causal_selection_score(row, args.early_margin_s)
                        all_causal.append(row)
        pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "early_warning_training_history.csv", index=False)
        pd.DataFrame(all_causal).to_csv(run_dir / "metrics" / "early_warning_causal_details.csv", index=False)

    causal = pd.DataFrame(all_causal)
    causal.to_csv(run_dir / "metrics" / "early_warning_causal_details.csv", index=False)
    summary = summarize_causal(all_causal, args.early_margin_s)
    summary.to_csv(run_dir / "metrics" / "early_warning_causal_summary.csv", index=False)
    val = summary[summary["split"] == "val"].copy()
    val["selection_score"] = (
        2.0 * val[f"early_{args.early_margin_s:.1f}s_recall_mean"]
        + 0.8 * val["pre_entry_recall_mean"]
        + 0.7 * val["event_precision_mean"]
        - 0.06 * val["false_alarms_per_min_mean"].clip(upper=20.0)
    )
    best_val = val.sort_values("selection_score", ascending=False).iloc[0].to_dict()
    test_best = summary[
        (summary["split"] == "test")
        & (summary["model"] == best_val["model"])
        & np.isclose(summary["threshold"].astype(float), float(best_val["threshold"]))
        & (summary["hysteresis"].astype(bool) == bool(best_val["hysteresis"]))
    ].iloc[0].to_dict()
    write_json(run_dir / "metrics" / "early_warning_best_selection.json", {"validation": best_val, "test": test_best})

    pred_best = next(p for p in all_predictions if p["model"].iloc[0] == best_val["model"])
    saved = make_failure_screenshots(run_dir, pred_best, best_val)

    lines = ["# Dedicated Early-Warning Experiment", ""]
    lines.append("This experiment trains directly for causal pre-entry warning instead of optimizing subclip AP. Frames closer than the required early margin are ignored or penalized so the model is not rewarded for only firing at the last moment.")
    lines.append("")
    lines.append(f"- Target mode: `{args.target_mode}`")
    lines.append(f"- Positive band: `{args.min_early_s:.2f}s <= time_to_entry <= {args.max_early_s:.2f}s`")
    lines.append(f"- Causal early metric: `>= {args.early_margin_s:.2f}s before physical entry`")
    lines.append(f"- Hard-negative weight: `{args.hard_negative_weight}`")
    lines.append("")
    lines.append("## Best Validation-Selected Test Result")
    lines.append("")
    lines.append("| model | threshold | hysteresis | pre-entry recall | early recall | precision | FA/min | median early s | detected | early | danger |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    lines.append(
        f"| {test_best['model']} | {float(test_best['threshold']):.2f} | {bool(test_best['hysteresis'])} | "
        f"{test_best['pre_entry_recall_mean']:.3f} | {test_best[f'early_{args.early_margin_s:.1f}s_recall_mean']:.3f} | "
        f"{test_best['event_precision_mean']:.3f} | {test_best['false_alarms_per_min_mean']:.3f} | "
        f"{test_best['median_early_warning_s_mean']:.3f} | {int(test_best['pre_entry_detected_total'])} | "
        f"{int(test_best[f'early_{args.early_margin_s:.1f}s_detected_total'])} | {int(test_best['danger_videos_total'])} |"
    )
    lines.append("")
    lines.append("## Top Test Rows By Early Recall And False Alarms")
    lines.append("")
    lines.append("| model | threshold | hysteresis | pre-entry recall | early recall | precision | FA/min |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|")
    view = summary[summary["split"] == "test"].copy()
    view = view.sort_values([f"early_{args.early_margin_s:.1f}s_recall_mean", "false_alarms_per_min_mean", "event_precision_mean"], ascending=[False, True, False]).head(20)
    for _, row in view.iterrows():
        lines.append(
            f"| {row['model']} | {float(row['threshold']):.2f} | {bool(row['hysteresis'])} | "
            f"{row['pre_entry_recall_mean']:.3f} | {row[f'early_{args.early_margin_s:.1f}s_recall_mean']:.3f} | "
            f"{row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} |"
        )
    lines.append("")
    lines.append(f"Saved causal failure screenshots: `{saved}` in `{run_dir / 'error_review' / 'causal_failures'}`")
    (run_dir / "early_warning_experiment_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(run_dir)
    print(run_dir / "early_warning_experiment_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train models directly for causal early warning before physical entry.")
    parser.add_argument("--sequence-run", default="runs/exp_071_physical_entry_seq60_focal")
    parser.add_argument("--base-run", default="runs/exp_070_physical_entry_baseline")
    parser.add_argument("--run-name", default="exp_093_early_warning_direct")
    parser.add_argument("--target-mode", choices=["band", "cumulative", "late_penalty"], default="band")
    parser.add_argument("--min-early-s", type=float, default=0.5)
    parser.add_argument("--max-early-s", type=float, default=1.5)
    parser.add_argument("--early-margin-s", type=float, default=0.5)
    parser.add_argument("--hard-negative-weight", type=float, default=2.5)
    parser.add_argument("--specs", nargs="+", default=["tcn_bce_aug:tcn:bce:aug", "tcn_focal_aug:tcn:focal:aug", "tcn_bce_noaug:tcn:bce:noaug", "cnn1d_bce_aug:cnn1d:bce:aug", "gru_bce_noaug:gru:bce:noaug", "cnn_gru_bce_aug:cnn_gru:bce:aug"])
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--patience", type=int, default=7)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--max-pos-weight", type=float, default=20.0)
    parser.add_argument("--persistence-windows", type=int, default=2)
    parser.add_argument("--seed", type=int, default=777)
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_093_early_warning_direct_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["early_warning_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
